For performing new tracing first import and make Tracer object based on config file settings.
The config file contains locations of images and ROIs for tracing, as well as settings for tracing, quality control and data output and saving.

If the analysis has already been run before, we can load the data instead:

In [1]:
import pandas as pd
import numpy as np
import os
import chromatin_tracing_python.image_processing_functions as ip
import chromatin_tracing_python.tracing_functions as tr
from chromatin_tracing_python.tracer_h5 import Tracer_decon

T=Tracer_decon(r"M:\ChromatinTeam\Images_processing\20200603_Exp328_KSB_Elyra_tracing_myc_D1-10\20200604_Exp_328_tracing_config_decon_h5.yaml")
wdir = r"M:\ChromatinTeam\Images_processing\20200603_Exp328_KSB_Elyra_tracing_myc_D1-10\04_TRACING"+os.sep
traces = pd.read_hdf(wdir+"20200603_Exp328_KSB_Elyra_tracing_myc_D1-10_v1_decon__traces.h5")
imgs=np.moveaxis(ip.read_tif_image(wdir+'20200603_Exp328_KSB_Elyra_tracing_myc_D1-10_v1_decon__imgs.tiff'),1,2)

traces=T.reapply_QC(traces)

ModuleNotFoundError: No module named 'chromatin_tracing_python.tracing_functions'

In [ ]:
with napari.gui_qt():
    viewer = napari.view_image(x[:,:,0], contrast_limits=(0,50000), blending='additive', colormap='magenta')
    viewer.add_image(x[:,:,1], contrast_limits=(0,2000), blending='additive', colormap='green')
    viewer.add_image(x[:,:,2], contrast_limits=(0,2000), blending='additive', colormap='blue')

In [ ]:
Once we have the traces loaded or calculated, we can look at the trace dataframe. It has a multilevel index with a unique trace ID as main index, and image name, nuclear ownership and frame nr as secondary index for each point. Each fitted point is documented with the full output of the gaussian fit, as well as a quality control metric defined from the config file. Units in nm.

In [ ]:
traces.query('trace_ID == 63')

We can easily visualize one or more traces by their trace_IDs now, either one or several (by a list of trace_IDs). To do this we load the tracing functions module that contains several helper functions for trace analysis.

In [ ]:
tr.plot_traces(traces,[31])

Let us get some more statistics on our traces:

In [ ]:
import plotly.express as px
px.histogram(traces.groupby(level='trace_ID').sum(), x='QC', width=500)

In [ ]:
px.bar(traces.groupby('frame').sum(), y='QC', width=500)

Visualize fits on top of actual image data using napari.

In [ ]:
import napari
points=tr.points_for_overlay(traces.query('QC == 1'))
with napari.gui_qt():
    viewer = napari.view_image(np.moveaxis(np.max(imgs,axis=2),0,1), contrast_limits=(100,10000))
    viewer.add_points(points[:,(0,1,3,4)], size=[0,0,1,1], face_color='red', symbol='cross', n_dimensional=True)

Depending on the quality of the dataset, some traces might be too short for a useful anaylsis. These can be filtered out from the traces and pair-wise distance matrix before analysing further. Once filtered, we can get a sense of the data from the average pwd matrix (missing distances are nan, so we have to use nanmean).

In [ ]:
traces_long = tr.tracing_length_qc(traces, min_length=9)
pwds_long = tr.pwd_calc(traces_long)

In [ ]:
print('Number of traces in analysis: ', pwds_long.shape[0])
pwds_mean=np.clip(np.round(-np.nanmedian(pwds_long, axis=0),0).astype(int),-300,0)

import plotly.figure_factory as ff
fig = ff.create_annotated_heatmap(pwds_mean,colorscale='hot')
fig.update_layout(
    width = 600,
    height = 600,
    yaxis_autorange="reversed"
)
fig.show()

Further, we can do a paired analysis on traces to figure out similarities between sets of traces. The output of this analysis contains the original indexes and point coordinates of the two compared indexes, the second trace aligned to the first by least squares fitting, and three metrics of similarity between the two traces. These are the mse of the alignment, and the mse of the two pair-wise distance matrices, as well as the Pearson's correlation coefficient between the PWDs. We sort the matrix for to find pairs with a high PCC.

In [ ]:
pairs = tr.trace_analysis(traces_long, pwds_long)

In [ ]:
pairs.sort_values(by=['aligned_mse'], ascending=True).head()

We can visualize the aligned traces by their index in the pair matrix.

In [ ]:
tr.plot_aligned_traces(traces,[30,31])

We can immediately use the paired similarity values to group the traces. We can use any of the three similarity metrics defined in the pairs analysis. This constructs a dendrogram showing simple hierichal grouping of the dataset. The labels in the dendrogram are the original trace_id of each trace.

In [ ]:
cluster_df = tr.trace_clustering(pairs, metric='aligned_pcc', method='average', color_threshold=0.5)

In [ ]:
cluster_id = 10
cluster_members = cluster_df[cluster_df['cluster']==cluster_id]['trace_ID'].values
print(cluster_members)

In [ ]:
clust_aligned, clust_mean = tr.general_procrustes_analysis(traces, cluster_members)
tr.plot_gpa_output(clust_aligned, clust_mean)

We can visualize these individually as before. To visualize pairs in the identified grouping, we can look up the index in the pairs analysis and plot these using the paired plotting function as well. Note that for lookups in the pairs dataframe the lower index is always idx1, the higher idx2.

In [ ]:
aligned_all_gpa=tr.run_gpa_all_clusters(traces_long, cluster_df, min_cluster=2)
tr.plot_multi_points(aligned_all_gpa)

In [ ]:
tr.radius_of_gyration(clust_mean)

In [ ]:
tr.elongation(clust_mean)